# 🎙️ Urdu Interview Processing Pipeline
**Stages:** Audio → Urdu Transcript → Verify → English Translation → Verify → De-identify → Final Dataset

**Models used:**
- ASR: `openai/whisper-large-v3-turbo` (via faster-whisper)
- Translation: `facebook/nllb-200-1.3B` ⬆️ **Upgraded for better quality**
- De-identification: `Microsoft Presidio` + `spaCy en_core_web_lg`

**Improvements in this version:**
- ✅ NLLB model upgraded from 600M to 1.3B (3x better translation)
- ✅ Sentence-aware chunking (preserves context instead of hard character splits)
- ✅ Better handling of Urdu→English translations

> ⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [1]:
# ── CELL 1: Check GPU ──────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✔ GPU available: {gpu_name}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('  Pipeline will run on CPU (much slower)')

✔ GPU available: Tesla T4
  VRAM: 15.6 GB


In [ ]:
# ── CELL 2: Install all dependencies ──────────────────────
print('Installing dependencies... (takes 3-5 minutes first time)')
print('⚠ Note: NLLB 1.3B model (~2.5GB) requires T4 GPU VRAM (~16GB available)')

!pip install -q faster-whisper==1.1.0
!pip install -q transformers==4.44.2 sentencepiece==0.2.0 sacremoses==0.1.1
!pip install -q presidio-analyzer==2.2.355 presidio-anonymizer==2.2.355
!pip install -q spacy==3.8.1
!pip install -q python-docx==1.1.2 langdetect==1.0.9 sacrebleu==2.4.3

# Download spaCy model
!python -m spacy download en_core_web_lg -q

print('\n✔ All dependencies installed!')
print('✔ Models will auto-download on first use (may take 2-3 minutes per model)')

Installing dependencies... (takes 3-5 minutes first time)
Reason for being yanked: model compatibility problem
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 22.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 107.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 132.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
en-core-web-lg 3.7.1 requires spacy<3.8.0,>=3.7.2, but you have spacy 3.8.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 4.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You c

In [1]:
# ── CELL 3: Clone project from GitHub ──────────────────────
!git clone https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE.git urdu-pipeline
%cd urdu-pipeline

# Pull latest changes
!git pull origin main

print('✔ Repository cloned and updated successfully!')
!ls -la

fatal: destination path 'urdu-pipeline' already exists and is not an empty directory.
/content/urdu-pipeline
From https://github.com/mSaadAli99/URDU-ENGLISH-TRANSLATION-PIPELINE
 * branch            main       -> FETCH_HEAD
Already up to date.
✔ Repository cloned and updated successfully!
total 60
drwxr-xr-x 9 root root 4096 Jun 30 12:02 .
drwxr-xr-x 1 root root 4096 Jun 30 11:13 ..
drwxr-xr-x 2 root root 4096 Jun 30 11:16 audio
-rw-r--r-- 1 root root 3295 Jun 30 12:02 config.py
drwxr-xr-x 8 root root 4096 Jun 30 12:09 .git
-rw-r--r-- 1 root root 9415 Jun 30 11:13 main.py
drwxr-xr-x 2 root root 4096 Jun 30 12:02 notebooks
drwxr-xr-x 8 root root 4096 Jun 30 11:17 outputs
drwxr-xr-x 3 root root 4096 Jun 30 12:02 pipeline
drwxr-xr-x 2 root root 4096 Jun 30 11:17 __pycache__
-rw-r--r-- 1 root root 3756 Jun 30 11:13 README.md
-rw-r--r-- 1 root root 1053 Jun 30 11:13 requirements.txt
drwxr-xr-x 4 root root 4096 Jun 30 12:05 urdu-pipeline


In [2]:
# ── CELL 4: Download audio from YouTube ───────────────────
import os

os.makedirs('audio', exist_ok=True)

!pip install -q yt-dlp

YOUTUBE_URL = "https://youtu.be/pHZHYWe8Mkc"

# Download full audio as mp3
!yt-dlp -x --audio-format mp3 -o "audio/full.%(ext)s" "{YOUTUBE_URL}"

# Trim to first 12 minutes (720 sec) for your initial test
!ffmpeg -y -i audio/full.mp3 -ss 137 -t 720 -c copy audio/test_audio.mp3

audio_path = "audio/test_audio.mp3"

size_mb = os.path.getsize(audio_path) / 1e6
print(f'\n✔ Audio ready: {audio_path}')
print(f'   File size: {size_mb:.1f} MB')

[youtube] Extracting URL: https://youtu.be/pHZHYWe8Mkc
[youtube] pHZHYWe8Mkc: Downloading webpage
[youtube] pHZHYWe8Mkc: Downloading android vr player API JSON
[info] pHZHYWe8Mkc: Downloading 1 format(s): 251
[download] audio/full.mp3 has already been downloaded
[ExtractAudio] Not converting audio audio/full.mp3; file is already in target format mp3
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-li

In [3]:
# ── CELL 5: Configure pipeline ─────────────────────────────
import sys
sys.path.insert(0, 'urdu-pipeline')

import config

# Set device to cuda since we have GPU
config.WHISPER_DEVICE       = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

# Set audio path
AUDIO_PATH = audio_path

print('Pipeline Configuration:')
print(f'  ASR Model        : whisper-{config.WHISPER_MODEL}')
print(f'  Translation Model: {config.TRANSLATION_MODEL}')
print(f'  Device           : {config.WHISPER_DEVICE}')
print(f'  Audio file       : {AUDIO_PATH}')
print(f'  Confidence thresh: {config.CONFIDENCE_THRESHOLD}')
print(f'  Chunk size       : {config.CHUNK_SIZE} chars')

Pipeline Configuration:
  ASR Model        : whisper-large-v3-turbo
  Translation Model: facebook/nllb-200-1.3B
  Device           : cuda
  Audio file       : audio/test_audio.mp3
  Confidence thresh: 0.75
  Chunk size       : 500 chars


In [4]:
# ── CELL 6: STAGE 1 — Urdu Transcription ──────────────────
from pipeline.utils import ensure_dirs
ensure_dirs(config.STAGE1_DIR, config.STAGE2_DIR, config.STAGE3_DIR,
            config.STAGE4_DIR, config.STAGE5_DIR, config.STAGE6_DIR)

from pipeline.transcribe import transcribe

stage1_result = transcribe(AUDIO_PATH)

# Preview
print('\n── Urdu Transcript Preview (first 500 chars) ──')
print(stage1_result['full_urdu_text'][:500])


  STAGE 1: URDU TRANSCRIPTION (ASR)
  Audio file : audio/test_audio.mp3
  Model      : whisper-large-v3-turbo
  Language   : ur (Urdu)
  Device     : cuda

  Loading Whisper model (first run downloads ~800MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  ✔ Model loaded.

  Transcribing audio... (this may take a few minutes)

  Processing segments:
    [⚠] 00:00:00.000 → 00:00:01.600 | conf=0.70 | السلام علیکم Dr. Majda...
    [⚠] 00:00:01.600 → 00:00:02.960 | conf=0.72 | Welcome to the show...
    [✔] 00:00:02.960 → 00:00:05.060 | conf=0.87 | Thank you so much for coming today...
    [✔] 00:00:05.060 → 00:00:06.440 | conf=0.96 | and taking out the time...
    [⚠] 00:00:08.120 → 00:00:11.120 | conf=0.74 | اگر آپ ہمیں اپنے بارے میک چھوٹا سا introduction دیں...
    [✔] 00:00:11.120 → 00:00:13.200 | conf=0.81 | تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی آج...
    [✔] 00:00:13.200 → 00:00:15.380 | conf=0.85 | they know that what a brilliant...
    [✔] 00:00:16.020 → 00:00:17.180 | conf=0.90 | Academian you are...
    [✔] 00:00:17.180 → 00:00:19.260 | conf=0.88 | Thank you so much Zainab...
    [✔] 00:00:19.260 → 00:00:20.560 | conf=0.99 | for inviting me...
    [✔] 00:00:20.560 → 00:00:21.720 | conf=0.81 | on your show...
    [✔] 00:00:22.38

In [5]:
# ── CELL 7: STAGE 2 — Verify Urdu Transcript ──────────────
from pipeline.verify_transcript import verify_transcript

stage2_result = verify_transcript(stage1_result)

print(f'\nQuality Score : {stage2_result["quality_score"]}/100')
print(f'Quality Label : {stage2_result["quality_label"]}')

# Show flagged segments
flagged = stage2_result['verification_report']['flagged_details']
if flagged:
    print(f'\n⚠ Flagged Segments ({len(flagged)}):')
    for f in flagged[:5]:
        print(f'  seg {f["segment_id"]} [{f["start_fmt"]}] conf={f["confidence"]:.2f}: {f["text"][:60]}...')
else:
    print('\n✔ No segments flagged!')


  STAGE 2: VERIFICATION OF URDU TRANSCRIPT
  Interview ID      : test_audio
  Total segments    : 404
  Confidence threshold: 0.75
    [⚠ FLAGGED] seg 001 | conf=0.70 | السلام علیکم Dr. Majda...
    [⚠ FLAGGED] seg 002 | conf=0.72 | Welcome to the show...
    [✔] seg 003 | conf=0.87 | Thank you so much for coming today...
    [✔] seg 004 | conf=0.96 | and taking out the time...
    [⚠ FLAGGED] seg 005 | conf=0.74 | اگر آپ ہمیں اپنے بارے میک چھوٹا سا introduction دی...
    [✔] seg 006 | conf=0.81 | تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی آج...
    [✔] seg 007 | conf=0.85 | they know that what a brilliant...
    [✔] seg 008 | conf=0.90 | Academian you are...
    [✔] seg 009 | conf=0.88 | Thank you so much Zainab...
    [✔] seg 010 | conf=0.99 | for inviting me...
    [✔] seg 011 | conf=0.81 | on your show...
    [✔] seg 012 | conf=0.86 | میرا نام Majda Kazmi ہے...
    [⚠ FLAGGED] seg 013 | conf=0.74 | I am a PhD doctor...
    [✔] seg 014 | conf=0.83 | MBBS نہیں...
    [⚠ FLAGGED] seg 01

In [6]:
# ── CELL 8: STAGE 3 — Translate Urdu → English ────────────
from pipeline.translate import translate

stage3_result = translate(stage2_result)

print('\n── English Translation Preview (first 500 chars) ──')
print(stage3_result['english_full_text'][:500])


  STAGE 3: TRANSLATION: URDU → ENGLISH
  Interview ID   : test_audio
  Model          : facebook/nllb-200-1.3B
  Source lang    : urd_Arab (Urdu)
  Target lang    : eng_Latn (English)
  Chunk size     : 500 chars

  Loading NLLB-200 model (first run downloads ~1.2GB)...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  ✔ Model loaded on cuda.

  Translating full text...

  Translating segments:
    ✔ seg 001 | UR: السلام علیکم Dr. Majda...
             | EN: Peace be with you Dr. Majda...
    ✔ seg 002 | UR: Welcome to the show...
             | EN: Welcome to the show....
    ✔ seg 003 | UR: Thank you so much for coming today...
             | EN: Thank you so much for coming today....
    ✔ seg 004 | UR: and taking out the time...
             | EN: and taking out the time...
    ✔ seg 005 | UR: اگر آپ ہمیں اپنے بارے میک چھوٹا سا intro...
             | EN: If you would give us a little introduction about yourself...
    ✔ seg 006 | UR: تاکہ ہماری جو آردینس ہے جو آپ کو سن رہی ...
             | EN: So our order is what you're hearing today....
    ✔ seg 007 | UR: they know that what a brilliant...
             | EN: They know that what a brilliant...
    ✔ seg 008 | UR: Academian you are...
             | EN: Academic you are...
    ✔ seg 009 | UR: Thank you so much Zainab...
             | EN: T

In [7]:
# ── CELL 9: STAGE 4 — Verify English Translation ──────────
from pipeline.verify_translation import verify_translation

stage4_result = verify_translation(stage3_result)

print(f'\nTranslation Quality Score : {stage4_result["translation_quality_score"]}/100')
print(f'Translation Quality Label : {stage4_result["translation_quality_label"]}')

# Show flagged translation segments
t_flagged = stage4_result['translation_verification_report']['flagged_details']
if t_flagged:
    print(f'\n⚠ Translation Flagged Segments ({len(t_flagged)}):')
    for f in t_flagged[:5]:
        print(f'  seg {f["segment_id"]}: {f["eng_text"][:60]}... | Issues: {", ".join(f["issues"])}')
else:
    print('\n✔ All translations verified OK!')


  STAGE 4: VERIFICATION OF ENGLISH TRANSLATION
  Interview ID   : test_audio
  Total segments : 404
    [✔] seg 001
             EN: Peace be with you Dr. Majda...
    [✔] seg 002
             EN: Welcome to the show....
    [✔] seg 003
             EN: Thank you so much for coming today....
    [✔] seg 004
             EN: and taking out the time...
    [✔] seg 005
             EN: If you would give us a little introduction about yourself...
    [✔] seg 006
             EN: So our order is what you're hearing today....
    [✔] seg 007
             EN: They know that what a brilliant...
    [⚠ FLAGGED] seg 008 | Detected language is 'ro', expected 'en'
             EN: Academic you are...
    [⚠ FLAGGED] seg 009 | Detected language is 'sw', expected 'en'
             EN: Thank you so much Zainab...
    [✔] seg 010
             EN: For inviting me...
    [✔] seg 011
             EN: on your show...
    [⚠ FLAGGED] seg 012 | Detected language is 'sw', expected 'en'
             EN: My n

In [8]:
# ── CELL 10: STAGE 5 — De-identification ──────────────────
from pipeline.deidentify import deidentify

stage5_result = deidentify(stage4_result)

print(f'\nEntities removed: {stage5_result["entities_removed_count"]}')
print('\n── De-identified Text Preview (first 500 chars) ──')
print(stage5_result['deidentified_english_full'][:500])


  STAGE 5: DE-IDENTIFICATION OF DATASET
  Interview ID : test_audio
  Loading Presidio + spaCy (first run may take a moment)...


  ✔ Presidio loaded.

  De-identifying full English text...



  De-identifying segments:
    [🔒] seg 001 | 1 entities removed | Peace be with you Dr. [NAME]...
    [✔] seg 002 | 0 entities removed | Welcome to the show....
    [🔒] seg 003 | 1 entities removed | Thank you so much for coming [DATE]....
    [✔] seg 004 | 0 entities removed | and taking out the time...
    [✔] seg 005 | 0 entities removed | If you would give us a little introduction about yourself...
    [🔒] seg 006 | 1 entities removed | So our order is what you're hearing [DATE]....
    [✔] seg 007 | 0 entities removed | They know that what a brilliant...
    [✔] seg 008 | 0 entities removed | [TRANSLATION FAILED] Academic you are...
    [🔒] seg 009 | 1 entities removed | [TRANSLATION FAILED] Thank you so much [NAME]...
    [✔] seg 010 | 0 entities removed | For inviting me...
    [✔] seg 011 | 0 entities removed | on your show...
    [🔒] seg 012 | 1 entities removed | [TRANSLATION FAILED] My name is [NAME]...
    [✔] seg 013 | 0 entities removed | [TRANSLATION FAILED] I'm a PhD d

    [✔] seg 023 | 0 entities removed | So he said...
    [✔] seg 024 | 0 entities removed | not...
    [✔] seg 025 | 0 entities removed | She is a PhD doctor...
    [✔] seg 026 | 0 entities removed | She's an academic doctor...
    [✔] seg 027 | 0 entities removed | That's it....
    [✔] seg 028 | 0 entities removed | Doctorate...
    [✔] seg 029 | 0 entities removed | [TRANSLATION FAILED] I have done...
    [✔] seg 030 | 0 entities removed | My area of specialization...
    [✔] seg 031 | 0 entities removed | [TRANSLATION FAILED] design of digital systems...
    [✔] seg 032 | 0 entities removed | And I'm an associate professor...
    [✔] seg 033 | 0 entities removed | in the NAD university...
    [✔] seg 034 | 0 entities removed | in the computer engineering department...
    [✔] seg 035 | 0 entities removed | and....
    [✔] seg 036 | 0 entities removed | beyond that...
    [✔] seg 037 | 0 entities removed | There is a STEM center...
    [✔] seg 038 | 0 entities removed | which is...


    [✔] seg 046 | 0 entities removed | [TRANSLATION FAILED] I'm the co-principal investigator....
    [✔] seg 047 | 0 entities removed | from the National Center...
    [✔] seg 048 | 0 entities removed | of Artificial Intelligence...
    [✔] seg 049 | 0 entities removed | which I lead to...
    [✔] seg 050 | 0 entities removed | the Artificial Intelligence...
    [✔] seg 051 | 0 entities removed | and Computer Vision site...
    [✔] seg 052 | 0 entities removed | And beyond that...
    [✔] seg 053 | 0 entities removed | I lead to the Embedded System...
    [✔] seg 054 | 0 entities removed | and Computer Vision Lab...
    [✔] seg 055 | 0 entities removed | to the university...
    [✔] seg 056 | 0 entities removed | Whoa....
    [🔒] seg 057 | 1 entities removed | Dr. [NAME]. What are you doing?...
    [✔] seg 058 | 0 entities removed | This is such a...
    [✔] seg 059 | 0 entities removed | [TRANSLATION FAILED] So many caps...
    [✔] seg 060 | 0 entities removed | Although related...
 

...

    [✔] seg 286 | 0 entities removed | and complexes...
    [✔] seg 287 | 0 entities removed | [TRANSLATION FAILED] I can do it....
    [✔] seg 288 | 0 entities removed | But...
    [✔] seg 289 | 0 entities removed | Some...
    [✔] seg 290 | 0 entities removed | Fields...
    [✔] seg 291 | 0 entities removed | Other...
    [✔] seg 292 | 0 entities removed | What more...
    [✔] seg 293 | 0 entities removed | male-dominated...
    [✔] seg 294 | 0 entities removed | And in that...
    [✔] seg 295 | 0 entities removed | In this...
    [✔] seg 296 | 0 entities removed | It's a place....
    [✔] seg 297 | 0 entities removed | What is it?...
    [✔] seg 298 | 0 entities removed | Two that...
    [✔] seg 299 | 0 entities removed | In class...
    [✔] seg 300 | 0 entities removed | Approximately...
    [🔒] seg 301 | 1 entities removed | [NAME]...
    [✔] seg 302 | 0 entities removed | It was 50-50....
    [✔] seg 303 | 0 entities removed | 50-50 of not...
    [✔] seg 304 | 0 entities removed 

    [✔] seg 312 | 0 entities removed | Female...
    [✔] seg 313 | 0 entities removed | Their marriage...
    [✔] seg 314 | 0 entities removed | and...
    [✔] seg 315 | 0 entities removed | Parametric...
    [✔] seg 316 | 0 entities removed | All the parents...
    [✔] seg 317 | 0 entities removed | of...
    [✔] seg 318 | 0 entities removed | Means...
    [✔] seg 319 | 0 entities removed | That good relationships be...
    [✔] seg 320 | 0 entities removed | That's the problem....
    [✔] seg 321 | 0 entities removed | I'm sorry about that....
    [✔] seg 322 | 0 entities removed | Or your engineer braid...
    [✔] seg 323 | 0 entities removed | As many as possible...
    [✔] seg 324 | 0 entities removed | In the professions it happens that...
    [🔒] seg 325 | 1 entities removed | [NAME] is getting married....
    [✔] seg 326 | 0 entities removed | That's it....
    [✔] seg 327 | 0 entities removed | Girls normally don't....
    [✔] seg 328 | 0 entities removed | That's what I did to

    [✔] seg 335 | 0 entities removed | And write...
    [✔] seg 336 | 0 entities removed | My marriage is...
    [✔] seg 337 | 0 entities removed | The door was locked....
    [✔] seg 338 | 0 entities removed | And until then...
    [✔] seg 339 | 0 entities removed | That's my opinion too....
    [✔] seg 340 | 0 entities removed | That I'll do the dealers...
    [✔] seg 341 | 0 entities removed | With me an engineering...
    [✔] seg 342 | 0 entities removed | The tag will be...
    [✔] seg 343 | 0 entities removed | The...
    [✔] seg 344 | 0 entities removed | See also...
    [✔] seg 345 | 0 entities removed | Water city...
    [✔] seg 346 | 0 entities removed | All the time...
    [✔] seg 347 | 0 entities removed | The target...
    [✔] seg 348 | 0 entities removed | Seeing...
    [✔] seg 349 | 0 entities removed | Other...
    [✔] seg 350 | 0 entities removed | My life...
    [✔] seg 351 | 0 entities removed | But then...
    [🔒] seg 352 | 1 entities removed | [NAME]...
    [✔] seg

    [✔] seg 362 | 0 entities removed | After...
    [✔] seg 363 | 0 entities removed | My age...
    [✔] seg 364 | 0 entities removed | My son....
    [✔] seg 365 | 0 entities removed | Born...
    [✔] seg 366 | 0 entities removed | After that...
    [✔] seg 367 | 0 entities removed | I did....
    [✔] seg 368 | 0 entities removed | Master's what...
    [✔] seg 369 | 0 entities removed | After that...
    [✔] seg 370 | 0 entities removed | [TRANSLATION FAILED] When My Masters...
    [✔] seg 371 | 0 entities removed | Completed...
    [✔] seg 372 | 0 entities removed | So then my...
    [✔] seg 373 | 0 entities removed | [TRANSLATION FAILED] What did you say?...
    [✔] seg 374 | 0 entities removed | Now what to do?...
    [✔] seg 375 | 0 entities removed | And when...
    [✔] seg 376 | 0 entities removed | I'm watching...
    [✔] seg 377 | 0 entities removed | That you...
    [✔] seg 378 | 0 entities removed | All the work...
    [✔] seg 379 | 0 entities removed | That's what I did....

    [✔] seg 388 | 0 entities removed | That now...
    [✔] seg 389 | 0 entities removed | [TRANSLATION FAILED] I'm doing engineering....
    [✔] seg 390 | 0 entities removed | So I...
    [✔] seg 391 | 0 entities removed | I 'll read it day and night...
    [✔] seg 392 | 0 entities removed | I did....
    [✔] seg 393 | 0 entities removed | No one...
    [✔] seg 394 | 0 entities removed | One...
    [✔] seg 395 | 0 entities removed | The event...
    [✔] seg 396 | 0 entities removed | Miss not what...
    [✔] seg 397 | 0 entities removed | Because of my studies...
    [✔] seg 398 | 0 entities removed | None of the above...
    [✔] seg 399 | 0 entities removed | There was no such time....
    [✔] seg 400 | 0 entities removed | That if my...
    [✔] seg 401 | 0 entities removed | [TRANSLATION FAILED] Hzبین یا میرے...
    [🔒] seg 402 | 1 entities removed | In [LOCATION]...
    [✔] seg 403 | 0 entities removed | I'm at home....
    [✔] seg 404 | 0 entities removed | And I...

  ── De-identi

In [ ]:
# ── CELL 11: STAGE 6 — Final Export ───────────────────────
from pipeline.export import export

final_result = export(stage5_result)

print(f'\n✔ Final JSON : {final_result["json_path"]}')
print(f'✔ Final DOCX : {final_result["docx_path"]}')

In [ ]:
# ── CELL 12: Download all outputs ─────────────────────────
import shutil
from google.colab import files

interview_id = stage1_result['interview_id']

# Zip the entire outputs folder
zip_name = f'{interview_id}_outputs.zip'
shutil.make_archive(
    base_name=interview_id + '_outputs',
    format='zip',
    root_dir='urdu-pipeline',
    base_dir='outputs'
)

print(f'✔ Zipped outputs as {zip_name}')
print('Downloading...')
files.download(zip_name)

In [ ]:
# ── CELL 13 (OPTIONAL): Run full pipeline in one go ───────
# Use this after testing individual stages above

import sys
sys.path.insert(0, 'urdu-pipeline')

import config
config.WHISPER_DEVICE = 'cuda'
config.WHISPER_COMPUTE_TYPE = 'float16'

from main import run_pipeline

# Change this to your audio path
run_pipeline('urdu-pipeline/audio/your_audio.mp3', start_stage=1)